# MONTE quick start

This tutorial walks through the following workflow:

1. Load the example methylation dataset
2. Train a MONTE model
    1. Manual model training
    2. Hyperparameter tuning
    3. Bayesian transfer learning
3. Estimate tumor purity
4. Correct methylation values

## 1. Load the example methylation dataset

In [1]:
from monte import load_example_data

In [2]:
X_train, X_val, X_test, y_train, y_val, y_test = load_example_data()

## 2. Train a MONTE model

### Manual model training

In [3]:
from monte import Monte

In [4]:
# create a model
model = Monte()

# fit the model on the training data, where X_train is the methylation beta matrix and y_train is the purity
model.fit(X_train, y_train)

### Hyperparameter tuning

Hyperparameter tuning can select the optimal number of top-ranked probes (`best_top_n`) for purity prediction. Because manually trained models do not support this parameter, we recommend using `train_with_cv` for model training.

In [5]:
from monte import train_with_cv

In [6]:
tuned_model = train_with_cv(X_train, y_train, n_splits=5)

In [7]:
tuned_model.best_top_n

10

### Bayesian transfer learning

MONTE could leverage existing or pretrained models on new datasets with different purity metrics.

In [8]:
from monte import fine_tune_with_cv

In [9]:
fine_tuned_model = fine_tune_with_cv(tuned_model, X_val, y_val)

## 3. Estimate tumor purity

To estimate the tumor purity, we could simply call the method, `predict_purity`.

In [10]:
predicted_purity = fine_tuned_model.predict_purity(X_test)

Compare the predicted purity with the ground truth in the test set.

In [11]:
from scipy.stats import pearsonr
print(f"Pearson correlation: {pearsonr(y_test, predicted_purity)[0]}")

Pearson correlation: 0.9347592409427484


## 4. Correct methylation values

To adjust beta values to a desired purity level, call `purify_values()` with the `target_purity` argument. In most cases, we recommend using `target_purity=1`; however, the method also allows you to specify any purity value you want.

In [12]:
purified_beta = fine_tuned_model.purify_values(X_test, target_purity=1)